# Mini Project: Meeting Notes Extractor
A CLI-style tool that takes a paragraph of meeting notes and returns structured JSON containing dates, names, and action items. Combines roles, multi-turn conversation, and structured outputs in one place.

## Step 1: Setup

**Main points:**
- Same setup as the core learning notebook: load the API key from `.env` and create the client
- Make sure your `.env` file is in the same folder as this notebook

In [ ]:
import os
import sys
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## Step 2: Define the Output Shape & System Prompt

**Main points:**
- Decide the exact JSON structure before writing extraction logic
- The system prompt tells the model what to extract, the exact keys to use, and to respond ONLY in JSON
- Including one example (few-shot) inside the prompt improves consistency

In [ ]:
SYSTEM_PROMPT = """You extract structured data from meeting notes.

Respond ONLY with valid JSON in exactly this format, nothing else:
{
  "dates": ["list of any dates mentioned, as written in the text"],
  "names": ["list of people's names mentioned"],
  "action_items": ["list of tasks or follow-ups, rephrased clearly as 'Person: action'"]
}

If a category has nothing, return an empty list for it. Do not add extra keys or commentary.

Example:
Input: "John and Sarah met on March 3rd. Sarah will send the report by Friday."
Output: {"dates": ["March 3rd", "Friday"], "names": ["John", "Sarah"], "action_items": ["Sarah: send the report"]}
"""

print(SYSTEM_PROMPT)

## Step 3: Extraction Function

**Main points:**
- Sends the system prompt + paragraph to the API with `response_format={"type": "json_object"}`
- `temperature=0` keeps extraction consistent rather than creative
- Parses the JSON response and fills in any missing keys with empty lists
- Returns the updated `messages` list so the conversation can continue (used for corrections later)

In [ ]:
def extract_meeting_data(paragraph, messages=None):
    if messages is None:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    messages.append({"role": "user", "content": paragraph})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})

    try:
        data = json.loads(reply)
        required_keys = ["dates", "names", "action_items"]
        for key in required_keys:
            if key not in data:
                data[key] = []
        return data, messages
    except json.JSONDecodeError:
        print("Model didn't return valid JSON. Raw output:")
        print(reply)
        return None, messages

## Step 4: Test With a Clean Example

**Main points:**
- Start with a simple, well-structured paragraph to confirm the pipeline works end to end

In [ ]:
test_paragraph = "John and Sarah met on March 3rd. Sarah will send the report by Friday."

data, messages = extract_meeting_data(test_paragraph)
print(json.dumps(data, indent=2))

## Step 5: Test With Messy, Realistic Input

**Main points:**
- Real meeting notes are rarely clean — test with no dates, indirect action items, and multiple names mixed together
- This is where you discover where the prompt needs tightening

In [ ]:
messy_paragraph = "We had a quick sync today. Alex mentioned the design review is still pending and someone should probably follow up with the vendor soon. No firm date was set."

data_messy, messages_messy = extract_meeting_data(messy_paragraph)
print(json.dumps(data_messy, indent=2))

## Step 6: Correction Loop (Multi-turn)

**Main points:**
- If the extraction misses something, send a follow-up message in the SAME conversation
- This works because `messages` already holds the full history, so the model has context on what it extracted the first time

In [ ]:
def request_correction(messages, instruction):
    """Send a follow-up correction in the same conversation."""
    messages.append({"role": "user", "content": instruction})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})

    try:
        return json.loads(reply), messages
    except json.JSONDecodeError:
        print("Correction also failed to parse:")
        print(reply)
        return None, messages

In [ ]:
# Example correction
corrected_data, messages_messy = request_correction(
    messages_messy,
    "You missed an action item about following up with the vendor. Please redo the extraction."
)
print(json.dumps(corrected_data, indent=2))

## Step 7: CLI Wrapper

**Main points:**
- Wraps everything into a runnable script: takes input via command-line argument or prompt, runs extraction, prints JSON, and allows optional corrections
- This cell is written as a script entry point — runs interactively when executed as a `.py` file, but works fine triggered manually in the notebook too

In [ ]:
def main():
    if len(sys.argv) > 1:
        paragraph = " ".join(sys.argv[1:])
    else:
        paragraph = input("Paste your meeting notes:\n")

    data, messages = extract_meeting_data(paragraph)

    if data:
        print(json.dumps(data, indent=2))
    else:
        print("Extraction failed.")
        return

    while True:
        fix = input("\nAnything to correct? (press Enter to skip, or describe the fix): ")
        if not fix.strip():
            break
        data, messages = request_correction(messages, fix)
        if data:
            print(json.dumps(data, indent=2))

# Uncomment to run interactively:
# main()